In [7]:
import pandas as pd
import numpy as np

# 1. Load the Core Files
results = pd.read_csv('data/MRegularSeasonDetailedResults.csv')
teams = pd.read_csv('data/MTeams.csv')
seeds = pd.read_csv('data/MNCAATourneySeeds.csv')

# 2. Clean Seeds: Convert 'W01' -> 1 (Integer)
# This allows the model to calculate "Seed Difference" mathematically.
seeds['Seed'] = seeds['Seed'].apply(lambda x: int(x[1:3]))

# 3. Create Season-Long Averages for every team
winning_stats = results.groupby(['Season', 'WTeamID'])[['WScore', 'WFGM', 'WAst']].mean().rename(
    columns={'WScore': 'Score', 'WFGM': 'FGM', 'WAst': 'Ast'})
losing_stats = results.groupby(['Season', 'LTeamID'])[['LScore', 'LFGM', 'LAst']].mean().rename(
    columns={'LScore': 'Score', 'LFGM': 'FGM', 'WAst': 'Ast'})

# Combine and group by Season/Team to get one row per team per year
team_stats = pd.concat([winning_stats, losing_stats]).groupby(level=[0, 1]).mean()

# 4. Filter for the most recent season stats to use for the 2026 Bracket
# This ensures we are predicting based on current team performance.
latest_season = team_stats.index.get_level_values(0).max()
latest_stats = team_stats.xs(latest_season, level=0)

print(f"✅ Step 1 Complete: Data prepared for Season {latest_season}.")

✅ Step 1 Complete: Data prepared for Season 2026.


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# 1. Merge Seeds into your Season Results
# We need to see what the seeds were for every game played in tournament history to train
train_results = results.merge(seeds, left_on=['Season', 'WTeamID'], right_on=['Season', 'TeamID'])
train_results = train_results.rename(columns={'Seed': 'WSeed'}).drop('TeamID', axis=1)
train_results = train_results.merge(seeds, left_on=['Season', 'LTeamID'], right_on=['Season', 'TeamID'])
train_results = train_results.rename(columns={'Seed': 'LSeed'}).drop('TeamID', axis=1)

# 2. Build the Features (Difference in Stats + Difference in Seeds)
side_a = pd.DataFrame()
side_a['ScoreDiff'] = train_results['WScore'] - train_results['LScore']
side_a['FGM_Diff'] = train_results['WFGM'] - train_results['LFGM']
side_a['Ast_Diff'] = train_results['WAst'] - train_results['LAst']
side_a['SeedDiff'] = train_results['WSeed'] - train_results['LSeed']
side_a['Result'] = 1

side_b = pd.DataFrame()
side_b['ScoreDiff'] = train_results['LScore'] - train_results['WScore']
side_b['FGM_Diff'] = train_results['LFGM'] - train_results['WFGM']
side_b['Ast_Diff'] = train_results['LAst'] - train_results['WAst']
side_b['SeedDiff'] = train_results['LSeed'] - train_results['WSeed']
side_b['Result'] = 0

train_df = pd.concat([side_a, side_b]).dropna()

# 3. Scaling & Final Training
scaler = StandardScaler()
features = ['ScoreDiff', 'FGM_Diff', 'Ast_Diff', 'SeedDiff']
X = scaler.fit_transform(train_df[features])
y = train_df['Result']

# We changed C from 0.01 to 0.001 to prevent over-confidence
model = LogisticRegression(C=0.001)
model.fit(X, y)

print("✅ Step 2 Updated: Model is now more conservative.")

✅ Step 2 Updated: Model is now more conservative.


In [9]:
# 3. THE FINAL NAME TRANSLATOR (Matches your Excel/Console exactly)
name_map = {
    "St. John's": "St John's",
    "Northern Iowa": "Northern Iowa",
    "Saint Mary's": "St Mary's CA",
    "Texas A&M": "Texas A&M",
    "Iowa State": "Iowa St",
    "Texas Tech": "Texas Tech",
    "Miami (FL)": "Miami FL",
    "Miami (OH)": "Miami OH",
    "UConn": "Connecticut",
    "Ohio State": "Ohio St",
    "Michigan State": "Michigan St",
    "North Dakota State": "N Dakota St",
    "Utah State": "Utah St",
    "Tennessee State": "Tennessee St",
    "Wright State": "Wright St",
    "Kennesaw State": "Kennesaw",
    "Long Island": "LIU Brooklyn",
    "Prairie View A&M": "Prairie View",
    "McNeese": "McNeese St",
    "Queens": "Queens NC",
    "Saint Louis": "St Louis"
}

# 2. THE PREDICTION FUNCTION
def predict_game_v4(team1_name, team2_name, s1_seed, s2_seed):
    t1_lookup = name_map.get(team1_name, team1_name)
    t2_lookup = name_map.get(team2_name, team2_name)

    try:
        t1_id = teams[teams['TeamName'] == t1_lookup]['TeamID'].values[0]
        t2_id = teams[teams['TeamName'] == t2_lookup]['TeamID'].values[0]
        s1 = team_stats.loc[(slice(None), t1_id), :].mean()
        s2 = team_stats.loc[(slice(None), t2_id), :].mean()
    except:
        return None

    # MIDDLE GROUND WEIGHT: Fixed Purdue while calming down UNC/BYU
    seed_weight = 3.0

    diff = pd.DataFrame([[
        s1['Score'] - s2['Score'],
        s1['FGM'] - s2['FGM'],
        s1['Ast'] - s2['Ast'],
        (s1_seed - s2_seed) * seed_weight
    ]], columns=['ScoreDiff', 'FGM_Diff', 'Ast_Diff', 'SeedDiff'])

    diff_scaled = scaler.transform(diff)
    return model.predict_proba(diff_scaled)[0][1]

print("✅ Step 3 Complete: Prediction engine is ready.")

✅ Step 3 Complete: Prediction engine is ready.


In [13]:
# %% [markdown]
# ## Official 2026 First Round Winners
# This section evaluates the model's predictive power by comparing its win
# probabilities against the actual 2026 tournament results.

# 1. THE OFFICIAL 2026 WINNERS
actual_winners = [
    "Duke", "TCU", "St. John's", "Kansas", "Louisville", "Michigan State", "UCLA", "UConn",
    "Arizona", "Utah State", "High Point", "Arkansas", "Texas", "Gonzaga", "Miami (FL)", "Purdue",
    "Michigan", "Saint Louis", "Texas Tech", "Alabama", "Tennessee", "Virginia", "Kentucky", "Iowa State",
    "Florida", "Iowa", "Vanderbilt", "Nebraska", "VCU", "Illinois", "Texas A&M", "Houston"
]

# 2. MATCHUP DATA
bracket_matchups = [
    ("Duke", "Siena", 1, 16), ("Ohio State", "TCU", 8, 9),
    ("St. John's", "Northern Iowa", 5, 12), ("Kansas", "Cal Baptist", 4, 13),
    ("Louisville", "South Florida", 6, 11), ("Michigan State", "North Dakota State", 3, 14),
    ("UCLA", "UCF", 7, 10), ("UConn", "Furman", 2, 15),
    ("Arizona", "Long Island", 1, 16), ("Villanova", "Utah State", 8, 9),
    ("Wisconsin", "High Point", 5, 12), ("Arkansas", "Hawaii", 4, 13),
    ("BYU", "Texas", 6, 11), ("Gonzaga", "Kennesaw State", 3, 14),
    ("Miami (FL)", "Missouri", 7, 10), ("Purdue", "Queens", 2, 15),
    ("Michigan", "Howard", 1, 16), ("Georgia", "Saint Louis", 8, 9),
    ("Texas Tech", "Akron", 5, 12), ("Alabama", "Hofstra", 4, 13),
    ("Tennessee", "Miami (OH)", 6, 11), ("Virginia", "Wright State", 3, 14),
    ("Kentucky", "Santa Clara", 7, 10), ("Iowa State", "Tennessee State", 2, 15),
    ("Florida", "Prairie View A&M", 1, 16), ("Clemson", "Iowa", 8, 9),
    ("Vanderbilt", "McNeese", 5, 12), ("Nebraska", "Troy", 4, 13),
    ("North Carolina", "VCU", 6, 11), ("Illinois", "Penn", 3, 14),
    ("Saint Mary's", "Texas A&M", 7, 10), ("Houston", "Idaho", 2, 15)
]

print(f"{'MATCHUP':<40} | {'CONFIDENCE'} | {'PREDICTED'} | {'ACTUAL'} | {'STATUS'}")
print("-" * 105)

correct_count = 0
total_games = 0

for i, (t1, t2, s1, s2) in enumerate(bracket_matchups):
    prob = predict_game_v4(t1, t2, s1, s2)

    if prob is not None:
        total_games += 1

        # Calculate Prediction & Confidence
        if prob >= 0.5:
            predicted_winner = t1
            confidence = prob
        else:
            predicted_winner = t2
            confidence = 1 - prob

        real_winner = actual_winners[i]
        status = "✅" if predicted_winner == real_winner else "❌"

        if predicted_winner == real_winner:
            correct_count += 1

        matchup_label = f"{t1} ({s1}) vs {t2} ({s2})"
        print(f"{matchup_label:<40} | {confidence:>9.2%} | {predicted_winner:<11} | {real_winner:<11} | {status}")

# 4. FINAL ACCURACY CALCULATION
print("-" * 105)
accuracy_pct = (correct_count / total_games) * 100
print(f"🎯 FINAL FIRST ROUND ACCURACY: {accuracy_pct:.2f}%")
print(f"Score: {correct_count} Correct / {total_games} Total")

MATCHUP                                  | CONFIDENCE | PREDICTED | ACTUAL | STATUS
---------------------------------------------------------------------------------------------------------
Duke (1) vs Siena (16)                   |    97.70% | Duke        | Duke        | ✅
Ohio State (8) vs TCU (9)                |    57.18% | Ohio State  | TCU         | ❌
St. John's (5) vs Northern Iowa (12)     |    88.07% | St. John's  | St. John's  | ✅
Kansas (4) vs Cal Baptist (13)           |    89.69% | Kansas      | Kansas      | ✅
Louisville (6) vs South Florida (11)     |    82.21% | Louisville  | Louisville  | ✅
Michigan State (3) vs North Dakota State (14) |    89.73% | Michigan State | Michigan State | ✅
UCLA (7) vs UCF (10)                     |    75.17% | UCLA        | UCLA        | ✅
UConn (2) vs Furman (15)                 |    93.90% | UConn       | UConn       | ✅
Arizona (1) vs Long Island (16)          |    96.30% | Arizona     | Arizona     | ✅
Villanova (8) vs Utah State (9)   

In [14]:
# --- ROUND OF 32 PREDICTIONS ---
# Matches confirmed as of March 21, 2026
round_32_matchups = [
    # East Region
    ("Duke", "TCU", 1, 9),
    ("St. John's", "Kansas", 5, 4),
    ("Louisville", "Michigan State", 6, 3),
    ("UCLA", "UConn", 7, 2),

    # West Region
    ("Arizona", "Utah State", 1, 9),
    ("Vanderbilt", "Nebraska", 5, 4),
    ("Miami (FL)", "Purdue", 7, 2),
    ("Texas", "Gonzaga", 11, 3),

    # Midwest Region
    ("Michigan", "Saint Louis", 1, 9),
    ("Texas Tech", "Alabama", 5, 4),
    ("Tennessee", "Virginia", 6, 3),
    ("Kentucky", "Iowa State", 7, 2),

    # South Region
    ("Florida", "Iowa", 1, 9),
    ("Arkansas", "High Point", 4, 12),
    ("VCU", "Illinois", 11, 3),
    ("Texas A&M", "Houston", 10, 2)
]

print(f"{'2026 ROUND OF 32 MATCHUP':<45} | WINNER PROBABILITY")
print("-" * 75)

for t1, t2, s1, s2 in round_32_matchups:
    # Assuming predict_game_v4 is defined in your environment
    prob = predict_game_v4(t1, t2, s1, s2)
    if prob is not None:
        winner = t1 if prob > 0.5 else t2
        win_p = prob if prob > 0.5 else (1 - prob)
        print(f"{t1} ({s1}) vs {t2} ({s2}):{'.'*15} | {winner} ({win_p:.2%})")

2026 ROUND OF 32 MATCHUP                      | WINNER PROBABILITY
---------------------------------------------------------------------------
Duke (1) vs TCU (9):............... | Duke (91.96%)
St. John's (5) vs Kansas (4):............... | Kansas (65.87%)
Louisville (6) vs Michigan State (3):............... | Michigan State (64.21%)
UCLA (7) vs UConn (2):............... | UConn (69.44%)
Arizona (1) vs Utah State (9):............... | Arizona (88.78%)
Vanderbilt (5) vs Nebraska (4):............... | Vanderbilt (56.30%)
Miami (FL) (7) vs Purdue (2):............... | Purdue (72.21%)
Texas (11) vs Gonzaga (3):............... | Gonzaga (90.03%)
Michigan (1) vs Saint Louis (9):............... | Michigan (87.13%)
Texas Tech (5) vs Alabama (4):............... | Alabama (56.99%)
Tennessee (6) vs Virginia (3):............... | Tennessee (53.24%)
Kentucky (7) vs Iowa State (2):............... | Iowa State (68.74%)
Florida (1) vs Iowa (9):............... | Florida (80.95%)
Arkansas (4) vs High P